[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/cs377/blob/main/supplemental/Neuron_Network_Pair.ipynb)

# From Neuron to Network: What Does a Hidden Layer Buy?
**CS 377 · companion to 18_2_3 From One Neuron to Many · pair activity**

**Driver:** this notebook is yours. Your partner, the **Recorder**, has the **From Neuron to Network quiz in Canvas** open, reads each question aloud and types your agreed answers.

**How the two fit together.** The quiz is the guide; this notebook is where you do the work for the quiz questions tagged **💻 NOTEBOOK**. Each section below is named after its quiz question (**Q3** here is **Q3** in the quiz). Questions tagged **💬 NO NOTEBOOK** have no section here: just talk them through.

**For each notebook question:** the code is written for you. The Recorder reads the question aloud; you find the matching section, make the small change marked **✏️** and run the cell (Shift+Enter). Read the result aloud and agree with your partner on what it means. Hints are under each section: try without them first, and open Hint 1 before Hint 2.

## Setup (run this first)
**Part 1 of the quiz (Q1-Q2) has no notebook work.** Run this cell while the Recorder reads Part 1.

It loads the Titanic data and makes what every section uses:
- `df`: the 887 passengers, with the features from 18_2_1 to 18_2_3 (`sex_f` is 1 for a woman; `pclass_2` and `pclass_3` are 1 for second and third class, so a first-class passenger has both at 0).
- `X_train`, `X_test`, `y_train`, `y_test`: the 18_2_3 Section 4 split, 665 training and 222 test passengers, with the seven features in `FEATURES`.
- `make_lr()` and `make_mlp()`: the 18_2_3 Section 4 logistic regression and shallow MLP (one hidden layer of 16 relu neurons), not yet trained.
- `X_xor`, `y_xor`: the four XOR points from 18_2_3 Section 2.
- `sigmoid(z)`: 1 / (1 + e^(-z)).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

pd.set_option('display.width', 140)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


# Titanic, with the same features as 18_2_1, 18_2_2 and 18_2_3
df = pd.read_csv('https://raw.githubusercontent.com/bsheese/cs377/main/data/data_titanic.csv')
df.columns = df.columns.str.lower().str.replace(' aboard', '', regex=False)
df['sibspouse'] = (df['siblings/spouses'] > 0).astype(int)
df['parentchild'] = (df['parents/children'] > 0).astype(int)
df['sex_f'] = (df['sex'] == 'female').astype(int)
df['fare_log'] = np.log1p(df['fare'])
df['pclass_2'] = (df['pclass'] == 2).astype(int)
df['pclass_3'] = (df['pclass'] == 3).astype(int)

THREE = ['sex_f', 'pclass_2', 'pclass_3']        # Q1 and Q3
FEATURES = ['age', 'sibspouse', 'parentchild', 'fare_log', 'sex_f', 'pclass_2', 'pclass_3']
X = df[FEATURES]
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


def make_lr():
    """The 18_2_3 Section 4 logistic regression: scale age and fare_log, pass the 0/1 columns through."""
    pre = ColumnTransformer([('num', StandardScaler(), ['age', 'fare_log'])], remainder='passthrough')
    return Pipeline([('pre', pre), ('clf', LogisticRegression(random_state=42, max_iter=1000))])


def make_mlp():
    """The 18_2_3 Section 4 shallow MLP: one hidden layer of 16 relu neurons, same preprocessing."""
    pre = ColumnTransformer([('num', StandardScaler(), ['age', 'fare_log'])], remainder='passthrough')
    return Pipeline([('pre', pre), ('clf', MLPClassifier(hidden_layer_sizes=(16,), activation='relu',
                                                         max_iter=1000, random_state=42))])


# XOR, from 18_2_3 Section 2
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

print(f"{len(df)} passengers   training: {len(y_train)}   test: {len(y_test)}")

# Part 2 (Q3-Q5): Checking It in the Notebook

## Q3 · The neuron formula against predict_proba
> **Quiz Q3:**
>
> The Q3 cell fits Q1's model and lists the six sex-and-class groups, with z for each group, `predict_proba`, and the observed survival rate and odds. The σ(z) column is missing: replace `np.nan` with the neuron formula and run it.
>
> - Does your σ(z) column match `predict_proba` in every row, and does the third-class women row match your Q1 answer?
> - For women in third class, which is higher, the model's probability or the observed survival rate, and by how much?
> - The model adds the same 2.529 to z for every woman, whatever her class, so it says being female multiplies the odds of survival by e^2.529 ≈ 12.5 in every class. The `observed odds` column is each group's observed rate / (1 - rate). Compute the observed odds ratio for first class (the women's observed odds divided by the men's) and for third class. Is one above 12.5 and the other below it?

**✏️ Your change:** Replace `np.nan` with `sigmoid(groups['z'])`.

**Tell your Recorder:** whether the columns match, the model's and the observed rate for third-class women, and the observed odds ratios for first and third class compared with 12.5.

<details><summary>Hint 1</summary>

Setup defined `sigmoid(z)`. It works on a whole column at once.

</details>

<details><summary>Hint 2</summary>

`groups['sigma(z)'] = sigmoid(groups['z'])`

</details>

In [ ]:
lr3 = LogisticRegression().fit(df[THREE], df['survived'])
b, w = lr3.intercept_[0], lr3.coef_[0]
print(f'b = {b:.3f}   ' + '   '.join(f'w for {f} = {v:.3f}' for f, v in zip(THREE, w)))

groups = pd.DataFrame({'sex_f':    [0, 0, 0, 1, 1, 1],
                       'pclass_2': [0, 1, 0, 0, 1, 0],
                       'pclass_3': [0, 0, 1, 0, 0, 1]},
                      index=['man, 1st', 'man, 2nd', 'man, 3rd', 'woman, 1st', 'woman, 2nd', 'woman, 3rd'])
groups['z'] = b + groups[THREE] @ w
groups['sigma(z)'] = np.nan      # ✏️ the neuron formula, applied to groups['z']
groups['predict_proba'] = lr3.predict_proba(groups[THREE])[:, 1]
groups['observed rate'] = df.groupby(THREE)['survived'].mean().loc[
    list(groups[THREE].itertuples(index=False, name=None))].values
groups['observed odds'] = groups['observed rate'] / (1 - groups['observed rate'])
groups['passengers'] = df.groupby(THREE).size().loc[
    list(groups[THREE].itertuples(index=False, name=None))].values
groups.round(3)

## Q4 · What a hidden layer needs for XOR
> **Quiz Q4:**
>
> The Q4 cell fits models to the four XOR points 10 times, with `random_state` 0 to 9, and counts how many of the 10 fits get all four points right. It also shows the predicted probabilities of class 1 from the fit with `random_state=0`. Every network has one hidden layer of 4 neurons, like 18_2_3 Section 2. Right now `ACTIVATIONS` holds only `'identity'`, which means no activation function: each hidden neuron outputs its weighted sum z unchanged. Change it to `['identity', 'tanh', 'relu']` and run it.
>
> - For logistic regression and for each activation, how many of the 10 `random_state` values solve XOR?
> - With `'identity'`, each hidden neuron outputs a weighted sum of x1 and x2, and the output neuron then takes a weighted sum of those four outputs. What kind of function of x1 and x2 is the output neuron's z, and why does that leave the network drawing a single straight line, like logistic regression?
> - Look at the `relu` row. What would you have concluded about relu from the `random_state=0` fit alone?

**✏️ Your change:** Change `ACTIVATIONS = ['identity']` to `ACTIVATIONS = ['identity', 'tanh', 'relu']`, then run the cell.

**Tell your Recorder:** how many random_state values solve XOR for each model, what kind of function z is with 'identity' and why that means one straight line, and what the relu row says about relying on one random_state.

<details><summary>Hint 1</summary>

`ACTIVATIONS` is a list; the cell adds one row to the table for each name in it.

</details>

<details><summary>Hint 2</summary>

`ACTIVATIONS = ['identity', 'tanh', 'relu']`

</details>

<details><summary>Hint 3</summary>

For the second bullet: write one hidden neuron's output as a·x1 + b·x2 + c. The output neuron's z is a weighted sum of four expressions like that, plus a bias. Collect the x1 terms and the x2 terms and see what form z has. Where z = 0 is the boundary between the classes.

</details>

In [ ]:
ACTIVATIONS = ['identity']      # ✏️ try ['identity', 'tanh', 'relu']

def xor_row(name, make_model):
    """Fit make_model(random_state) to the four XOR points for random_state 0-9."""
    solved = 0
    for rs in range(10):
        model = make_model(rs).fit(X_xor, y_xor)
        solved += int(model.score(X_xor, y_xor) == 1.0)
        if rs == 0:
            p0 = model.predict_proba(X_xor)[:, 1]
    row = {'model': name, 'random_states solved (of 10)': solved}
    for point, p in zip(['(0,0)', '(0,1)', '(1,0)', '(1,1)'], p0):
        row[f'P(1) at {point}'] = round(p, 2)
    return row

print('P(1) columns: predicted probability of class 1 from the random_state=0 fit')
rows = [xor_row('logistic regression', lambda rs: LogisticRegression(random_state=rs))]
for act in ACTIVATIONS:
    rows.append(xor_row(f'MLP (4,), {act}', lambda rs: MLPClassifier(
        hidden_layer_sizes=(4,), activation=act, solver='lbfgs', max_iter=10000, random_state=rs)))
pd.DataFrame(rows).set_index('model')

## Q5 · What the MLP gains and what it loses
> **Quiz Q5:**
>
> The Q5 cell fits the 18_2_3 Section 4 logistic regression (seven features) and the shallow MLP (one hidden layer of 16 relu neurons) on the 18_2_3 train/test split. To measure what being female does in each model, it makes two copies of the 222 test passengers, one where everyone is a woman and one where everyone is a man, and divides each passenger's odds of survival as a woman by their odds as a man. That ratio is the odds ratio for `sex_f` for that passenger. Right now the second copy sets `sex_f` to 1 as well, so every ratio is 1. Change it to 0 and run it.
>
> The logistic regression's odds ratio here (12.28) is not quite Q3's 12.5, because this model has seven features instead of three.
>
> - The logistic regression gives every passenger the same odds ratio. Use the neuron formula from Q1 to explain why.
> - What are the smallest and largest odds ratios the MLP gives, and which class has the smallest median? Connect that to what you found in Q3.
> - Name two things 18_2_1 and 18_2_2 let you report about `sex_f` from the logistic regression that the MLP cannot give you.

**✏️ Your change:** In the line marked ✏️, change `men['sex_f'] = 1` to `men['sex_f'] = 0`, then run the cell.

**Tell your Recorder:** why the logistic regression's odds ratio is the same for everyone, the MLP's smallest and largest odds ratio and the class with the smallest median, and two things the MLP cannot give you.

<details><summary>Hint 1</summary>

The ratio compares each passenger as a woman (`sex_f` = 1) with the same passenger as a man.

</details>

<details><summary>Hint 2</summary>

`men['sex_f'] = 0`

</details>

In [ ]:
lr = make_lr().fit(X_train, y_train)
mlp = make_mlp().fit(X_train, y_train)
for name, model in [('logistic regression', lr), ('MLP (16,)', mlp)]:
    correct = (model.predict(X_test) == y_test).sum()
    print(f'{name:<20} {correct} of {len(y_test)} test passengers right   accuracy {correct / len(y_test):.3f}')

women = X_test.copy()
women['sex_f'] = 1
men = X_test.copy()
men['sex_f'] = 1      # ✏️ make every passenger in this copy a man

def odds(p):
    return p / (1 - p)

pclass = np.select([X_test['pclass_2'] == 1, X_test['pclass_3'] == 1], ['2nd', '3rd'], '1st')
ratios = pd.concat([
    pd.DataFrame({'model': name, 'class': pclass,
                  'odds ratio': odds(m.predict_proba(women)[:, 1]) / odds(m.predict_proba(men)[:, 1])})
    for name, m in [('logistic regression', lr), ('MLP (16,)', mlp)]])

ax = sns.stripplot(data=ratios, x='class', y='odds ratio', hue='model', dodge=True,
                   order=['1st', '2nd', '3rd'], alpha=0.6)
ax.set_yscale('log')
ax.set_title('Odds ratio for sex_f, one dot per test passenger')
plt.show()
ratios.groupby(['model', 'class'])['odds ratio'].agg(['min', 'median', 'max', 'size']).round(2)

# Extra credit (Q6-Q7): A Network You Can Follow, and More Splits
Only if you have finished Q1-Q5.

## ⭐ Q6 · A hand-built network for XOR
> **Quiz Q6:**
>
> Here is a network for XOR with weights chosen by hand, so you can follow every number. It has two hidden neurons with ReLU, where ReLU(z) = max(0, z):
>
> - h1 = ReLU(x1 + x2)
> - h2 = ReLU(x1 + x2 - 1)
> - output: z = 10h1 - 20h2 - 5, and p = σ(z)
>
> First fill in h1, h2, z and p for the four XOR points on paper. The Q6 cell computes the same table, except that it leaves out the ReLU on h2. Run it as it is, then wrap h2 in `relu(...)` and run it again.
>
> - Does the corrected table match yours? Which points get p above 0.5?
> - Plot the four points at (h1, h2) instead of (x1, x2). Can a straight line separate the classes there?
> - How many of the four points does the version without the second ReLU get right? Connect that to Q2.

**✏️ Your change:** Run the cell once as it is and count how many points end up on the correct side of p = 0.5. Then change `h2 = x1 + x2 - 1` to `h2 = relu(x1 + x2 - 1)` and run it again.

**Tell your Recorder:** whether the table matches, which points get p above 0.5, whether a line separates the classes at (h1, h2), and how many points the version without the second ReLU gets right.

<details><summary>Hint 1</summary>

`relu` is defined at the top of the cell; it works on the whole array.

</details>

<details><summary>Hint 2</summary>

`h2 = relu(x1 + x2 - 1)`

</details>

In [ ]:
def relu(z):
    return np.maximum(0, z)

x1, x2 = X_xor[:, 0], X_xor[:, 1]
h1 = relu(x1 + x2)
h2 = x1 + x2 - 1      # ✏️ pass this through relu(...) too
z = 10 * h1 - 20 * h2 - 5
pd.DataFrame({'x1': x1, 'x2': x2, 'h1': h1, 'h2': h2, 'z': z,
              'p': sigmoid(z).round(4), 'label': y_xor})

## ⭐ Q7 · One split or ten?
> **Quiz Q7:**
>
> Q5 compared the two models on one train/test split, and 18_2_3 says that a gap of a couple of accuracy points on 222 test passengers could be noise. The Q7 cell refits both Q5 models on several different splits. Right now `SPLITS` holds only 42, the 18_2_3 split. Change it to `list(range(10))` and run it.
>
> - On how many of the 10 splits does the MLP get more test passengers right, and what is the mean difference in accuracy?
> - Does that change how you read Q5's gap? Would it change which model you would use if you also had to report the effect of being female?

**✏️ Your change:** Change `SPLITS = [42]` to `SPLITS = list(range(10))`, then run the cell.

**Tell your Recorder:** how many splits the MLP wins and the mean difference, then how you read Q5's gap and which model you would use.

<details><summary>Hint 1</summary>

`SPLITS` is a list of `random_state` values for `train_test_split`; the cell adds one row for each.

</details>

<details><summary>Hint 2</summary>

`SPLITS = list(range(10))`

</details>

In [ ]:
SPLITS = [42]      # ✏️ try list(range(10))

rows = []
for split in SPLITS:
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, random_state=split, stratify=y)
    lr_right = (make_lr().fit(Xa, ya).predict(Xb) == yb).sum()
    mlp_right = (make_mlp().fit(Xa, ya).predict(Xb) == yb).sum()
    rows.append({'split': split, 'LR right': lr_right, 'MLP right': mlp_right,
                 'MLP minus LR': mlp_right - lr_right,
                 'accuracy difference': (mlp_right - lr_right) / len(yb)})
res = pd.DataFrame(rows).set_index('split')
print(f"MLP more right on {(res['MLP minus LR'] > 0).sum()} of {len(res)} splits, "
      f"fewer on {(res['MLP minus LR'] < 0).sum()};   "
      f"mean accuracy difference {res['accuracy difference'].mean():.3f}")
res.round(3)